###===============================================================================
### Title: Canonicalisation and Semantic Clustering
### Author(s): Joffrey Marchi et al.
### Objective: Deduplicate orthographic variants using TF-IDF vectorisation and cluster semantic families using the DBSCAN algorithm.
### Inputs: data/intermediate/models_cleaned_deduplicated_gemma3_27b.xlsx
### Outputs: data/intermediate/models_clustered_gemma3_27b.xlsx
###===============================================================================

In [4]:
#-------------------------------------------------------------------------------
#### 0 - Library ####
#-------------------------------------------------------------------------------
import re, unicodedata
from unidecode import unidecode
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from rapidfuzz import fuzz
from collections import defaultdict
import numpy as np
from scipy.sparse.csgraph import connected_components
from datetime import datetime
from natsort import natsorted, ns

In [3]:
#-------------------------------------------------------------------------------
#### I - DATA IMPORT ####
#-------------------------------------------------------------------------------
base = pd.read_excel("data/intermediate/models_cleaned_deduplicated_gemma3_27b.xlsx")

base["original"] = base["normalized_names"]
base["norm"] = base["normalized_names"]
df = base[["original", "norm"]].copy()

FileNotFoundError: [Errno 2] No such file or directory: 'data/intermediate/models_cleaned_deduplicated_gemma3_27b.xlsx'

In [ ]:
#-------------------------------------------------------------------------------
#### II - VECTORISATION AND DEDUPLICATION (CHAR N-GRAMS) ####
#-------------------------------------------------------------------------------
# Core process for orthographic standardisation using character sub-parts
vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),  # Examines blocks of 3 to 6 letters. Highly robust to typos.
    min_df=1
)
X = vec.fit_transform(df["norm"])

# "Fuzzy" deduplication to group spelling mistakes
# Very strict radius (0.3) to group only highly similar spellings
nbrs = NearestNeighbors(metric="cosine", radius=0.3)
nbrs.fit(X)
adj = nbrs.radius_neighbors_graph(X, mode="connectivity")

# Identifying connected components
n_comp, labels_dup = connected_components(adj, directed=False)
df["dedupe_id"] = labels_dup

In [ ]:
#-------------------------------------------------------------------------------
#### III - CONDENSATION PRIOR TO DBSCAN ####
#-------------------------------------------------------------------------------
# Retaining a single canonical row per group of variants, weighted by frequency
unique_groups = df.groupby("dedupe_id").indices

unique_indices = [] # Rows retained for clustering
weights = []        # The number of variants they represent
ids_map = []        # Mapping back to original IDs

# Choice of the canonical form (the cleanest/most frequent) for display
canonical_map = {}

for did, indices in unique_groups.items():
    w = len(indices)
    subset = df.iloc[indices]
    
    # Selecting the shortest string as the "cleanest" canonical representation
    canon_text = subset.sort_values("norm", key=lambda x: x.str.len()).iloc[0]["norm"]
    canonical_map[did] = canon_text
    
    rep_idx = subset.index[0] 
    
    unique_indices.append(rep_idx)
    weights.append(w)
    ids_map.append(did)

df["canonical"] = df["dedupe_id"].map(canonical_map)

# Reduced matrix excluding quasi-duplicates
X_reduced = X[unique_indices]
sample_weights = np.array(weights)

print(f"Reduction : {X.shape[0]} phrases -> {X_reduced.shape[0]} unique concepts (corrected spelling)")

In [ ]:
#-------------------------------------------------------------------------------
#### IV - SEMANTIC CLUSTERING (DBSCAN) ####
#-------------------------------------------------------------------------------
# Expanding epsilon to group semantic families rather than just orthographic errors
clu = DBSCAN(eps=0.4, min_samples=4, metric="cosine")

# Integrating weights to allow small but highly frequent groups to form clusters
labels_reduced = clu.fit_predict(X_reduced, sample_weight=sample_weights)

# Re-mapping cluster labels to the original dataframe
dedupe_to_cluster = dict(zip(ids_map, labels_reduced))
df["cluster_id"] = df["dedupe_id"].map(dedupe_to_cluster)

clusters = defaultdict(list)
for _, r in df.iterrows():
    clusters[r["cluster_id"]].append(r["original"])

unique_clusters = defaultdict(list)
for dedupe_id, cluster_label in zip(ids_map, labels_reduced):
    concept_name = canonical_map[dedupe_id]
    unique_clusters[cluster_label].append(concept_name)

clustersAvantReclust = pd.DataFrame.from_dict(clusters, orient='index').transpose().add_prefix("Cluster ")
unique_clustersAll = pd.DataFrame.from_dict(unique_clusters, orient='index').transpose().add_prefix("Cluster ")
dfDedubAvant = df.drop_duplicates(subset=['canonical'])
textMiningAvant = df.copy()

In [ ]:
#-------------------------------------------------------------------------------
#### V - SUB-CLUSTERING FOR PROBLEMATIC GROUPS ####
#-------------------------------------------------------------------------------
def subcluster_group(df_source, target_cluster_id, new_eps=None, use_agglomerative=False, n_clusters=None):
    """
    Isolates a cluster, recalculates local TF-IDF, and re-fragments it.
    """
    mask = df_source["cluster_id"] == target_cluster_id
    df_sub = df_source[mask].copy()
    
    if df_sub.empty:
        return df_source
        
    vec_sub = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=1)
    X_sub = vec_sub.fit_transform(df_sub["norm"])
    
    if use_agglomerative and n_clusters:
        # AgglomerativeClustering forces classification and removes noise
        clu_sub = AgglomerativeClustering(n_clusters=n_clusters, metric='cosine', linkage='average')
        sub_labels = clu_sub.fit_predict(X_sub.toarray())
    else:
        # Stricter DBSCAN
        clu_sub = DBSCAN(eps=new_eps, min_samples=2, metric="cosine")
        sub_labels = clu_sub.fit_predict(X_sub)
        
    prefix = "noise" if target_cluster_id == -1 else str(target_cluster_id)
    new_cluster_names = [f"{prefix}_{lbl}" if lbl != -1 else f"{prefix}_noise" for lbl in sub_labels]
    
    df_source.loc[mask, "cluster_id"] = new_cluster_names
    return df_source

# A. Breaking down the mega-cluster 2 using a stricter DBSCAN epsilon
df = subcluster_group(df, target_cluster_id=2, new_eps=0.2)

# B. Organising the noise from cluster -1 using forced Agglomerative Clustering
df = subcluster_group(df, target_cluster_id=-1, use_agglomerative=True, n_clusters=20)

In [ ]:
#-------------------------------------------------------------------------------
#### VI - DICTIONARY RECALCULATION & EXPORT ####
#-------------------------------------------------------------------------------
dfDedub = df.drop_duplicates(subset=['canonical'])

clusters = defaultdict(list)
for _, r in df.iterrows():
    clusters[r["cluster_id"]].append(r["original"])

unique_clusters = defaultdict(list)
for _, r in dfDedub.iterrows():
    unique_clusters[r["cluster_id"]].append(r["canonical"])

dataCluster = pd.DataFrame.from_dict(clusters, orient='index').transpose().add_prefix("Cluster ")
Dataunique_clusters = pd.DataFrame.from_dict(unique_clusters, orient='index').transpose().add_prefix("Cluster ")
dfDedub = df.drop_duplicates(subset=['canonical'])

nom_fichier_export = "data/intermediate/models_clustered_gemma3_27b.xlsx"
print(f"\nSaving to file : {nom_fichier_export}...")

with pd.ExcelWriter(nom_fichier_export) as writer:  
    base.to_excel(writer, sheet_name='PubMed-gemma3_27b', index=False)
    textMiningAvant.to_excel(writer, sheet_name='TextMiningAllModelBeforeReclust', index=False)
    dfDedubAvant.to_excel(writer, sheet_name='DedupBaseAllModelBeforeReclust', index=False)
    clustersAvantReclust[natsorted(clustersAvantReclust.columns, alg=ns.SIGNED)].to_excel(writer, sheet_name='ClusterAllModelBeforeReclust', index=False)
    unique_clustersAll[natsorted(unique_clustersAll.columns, alg=ns.SIGNED)].to_excel(writer, sheet_name='ClusterDedupModelBeforeReclust', index=False)
    df.to_excel(writer, sheet_name='TextMiningAllModelAfterReclust', index=False)
    dfDedub.to_excel(writer, sheet_name='DedupBaseAllModelAfterReclust', index=False)
    dataCluster[natsorted(dataCluster.columns, alg=ns.SIGNED)].to_excel(writer, sheet_name='ClusterAllModelAfterReclust', index=False)
    Dataunique_clusters[natsorted(Dataunique_clusters.columns, alg=ns.SIGNED)].to_excel(writer, sheet_name='ClusterDedupModelAfterReclust', index=False)

print("Excel export completed successfully!")